# Experiment 3: Regression Analysis using Linear and Regularized Models

```
experiment3_loan_regression.py
================================
ICS1512 - Machine Learning Algorithms Laboratory
Experiment 3: Regression Analysis using Linear and Regularized Models

Uses the reusable module ml_lab_utils.py (from Experiment 1) for:
    - EDA                       -> generate_eda_summary()
    - Regression train/eval     -> train_evaluate_regression()
    - Regression metrics        -> regression_performance_metrics()
    - Global plot style         -> set_plot_style()

Dataset: Loan Amount Prediction (Analytics Vidhya / Kaggle "Predict Loan
Amount Data" family), 614 loan applications, mixed numeric/categorical
features, target = LoanAmount (continuous, in thousands).
```

## Reusable utilities (`ml_lab_utils`, from Experiment 1)

Inlined here so this notebook runs on its own without a separate `ml_lab_utils.py`.

In [ ]:
"""
ml_lab_utils.py
================
ICS1512 - Machine Learning Algorithms Laboratory
Reusable utility module used across ALL experiments.

Implements (per lab manual, Section 4):
    1. One reusable EDA function            -> generate_eda_summary()
    2. One reusable Regression function      -> train_evaluate_regression()
    3. One reusable Classification function  -> train_evaluate_classification()
    4. One reusable Regression metrics fn    -> regression_performance_metrics()
    5. One reusable Classification metrics   -> classification_performance_metrics()

Formatting rules enforced everywhere (per lab manual, Section 1):
    - Times New Roman, 15 pt for all text / legends
    - Bold, Times New Roman, 15 pt axis labels
    - Figures exported as .eps at 600 DPI (Section 3)

NOTE on fonts: "Times New Roman" itself is a proprietary Microsoft font and is
not installable on Linux. Liberation Serif is metrically-compatible (identical
glyph widths/kerning) and is registered here under the family name
"Times New Roman" so that rcParams['font.family'] = 'Times New Roman' works
transparently. On Windows/macOS, if the real Times New Roman is installed,
matplotlib will simply use that instead.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# 1. GLOBAL PLOT STYLE  (Section 1 of the manual)
# --------------------------------------------------------------------------
def set_plot_style(font_size=15):
    """
    Applies the mandatory lab formatting to every matplotlib figure:
        - Times New Roman (or metric-compatible Liberation Serif) font
        - 15 pt base font size
        - 15 pt Times New Roman legends
        - Bold, 15 pt, Times New Roman axis labels
    Call this once at the start of a notebook / script.
    """
    # Register Liberation Serif under the alias "Times New Roman" if the
    # genuine font is not present on this machine.
    installed_fonts = {f.name for f in fm.fontManager.ttflist}
    if "Times New Roman" not in installed_fonts:
        liberation_paths = [
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Italic.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-BoldItalic.ttf",
        ]
        for p in liberation_paths:
            if os.path.exists(p):
                fm.fontManager.addfont(p)
                # Force the registered family name to "Times New Roman"
                # (FontEntry is a frozen dataclass in modern matplotlib, so we
                # replace the last-added entry rather than mutate it in place)
                last = fm.fontManager.ttflist[-1]
                fm.fontManager.ttflist[-1] = fm.FontEntry(
                    fname=last.fname, name="Times New Roman",
                    style=last.style, variant=last.variant,
                    weight=last.weight, stretch=last.stretch, size=last.size,
                )

    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": font_size,
        "legend.fontsize": font_size,
        "legend.title_fontsize": font_size,
        "axes.labelsize": font_size,
        "axes.labelweight": "bold",
        "axes.titlesize": font_size,
        "axes.titleweight": "bold",
        "xtick.labelsize": font_size - 2,
        "ytick.labelsize": font_size - 2,
        "figure.titlesize": font_size + 2,
        "savefig.dpi": 600,
        "figure.dpi": 150,   # screen preview; export always forced to 600 (see save)
        "svg.fonttype": "none",
    })


def _bold_axis_labels(ax, xlabel=None, ylabel=None, title=None, fs=15):
    """Helper: apply Times New Roman / Bold / 15pt to a single axis explicitly."""
    fp_bold = fm.FontProperties(family="Times New Roman", weight="bold", size=fs)
    fp_reg = fm.FontProperties(family="Times New Roman", size=fs - 2)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontproperties=fp_bold)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontproperties=fp_bold)
    if title is not None:
        ax.set_title(title, fontproperties=fp_bold, fontsize=fs)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(fp_reg)
    leg = ax.get_legend()
    if leg is not None:
        for txt in leg.get_texts():
            txt.set_fontproperties(fp_reg)


def _save_eps(fig, save_path, also_png=True):
    """Export a figure as .eps at 600 DPI (Section 3 of the manual).

    If also_png is True, an additional .png copy is saved alongside the .eps
    (same basename) purely so the figure can be embedded when compiling the
    LaTeX report with pdflatex/xelatex, which cannot rasterize .eps directly
    without Ghostscript. The .eps remains the official, mandated deliverable.
    """
    if save_path is None:
        return None
    if not save_path.lower().endswith(".eps"):
        save_path = os.path.splitext(save_path)[0] + ".eps"
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig.savefig(save_path, format="eps", dpi=600, bbox_inches="tight")
    if also_png:
        png_path = os.path.splitext(save_path)[0] + ".png"
        fig.savefig(png_path, format="png", dpi=200, bbox_inches="tight")
    return save_path


# --------------------------------------------------------------------------
# 2. GENERIC EDA FUNCTION  (Section 4.1)  -> ONE consolidated 12-subplot figure
# --------------------------------------------------------------------------
def generate_eda_summary(df, target_col=None, dataset_name="Dataset",
                          save_path=None, figsize=(22, 16)):
    """
    Generic, reusable EDA function that works on ANY tabular dataset
    (classification, regression, or unlabeled). Produces ONE consolidated
    figure containing 12 EDA subplots on a single page, per Section 2 of the
    lab manual.

    Parameters
    ----------
    df : pandas.DataFrame
        The full dataset (features + target, if any).
    target_col : str or None
        Name of the target/label column, if present. If None, the function
        treats the dataset as unlabeled and adapts the 12-panel layout
        accordingly (no class-distribution / target-correlation panels).
    dataset_name : str
        Used in the figure's suptitle.
    save_path : str or None
        If given, the figure is exported as .eps @ 600 DPI to this path.
    figsize : tuple
        Overall figure size in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    set_plot_style()
    df = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)

    is_classification_target = (
        target_col is not None and
        (df[target_col].dtype == "object" or df[target_col].nunique() <= 20)
    )

    # Pick the most "informative" numeric feature (highest variance) as the
    # representative single feature for panels 6/8/9/10, instead of blindly
    # using the first column (which can be degenerate/constant, e.g. corner
    # pixels in an image dataset such as MNIST/Digits).
    if numeric_cols:
        # Prefer genuinely continuous columns (more than 5 distinct values) so
        # binary/near-constant encoded columns (e.g. a 0/1 "sex" flag, or
        # constant corner pixels in image data) are not picked as the
        # representative single feature for panels 6/8/9/10.
        continuous_cols = [c for c in numeric_cols if df[c].nunique() > 5]
        candidate_cols = continuous_cols if continuous_cols else numeric_cols
        variances = df[candidate_cols].var().sort_values(ascending=False)
        top_var_cols = variances.index.tolist()
        feat_a = top_var_cols[0]
        feat_b = top_var_cols[1] if len(top_var_cols) > 1 else top_var_cols[0]
        kde_cols = top_var_cols[:4]
    else:
        feat_a = feat_b = None
        kde_cols = []

    fig = plt.figure(figsize=figsize)
    fig.suptitle(f"Exploratory Data Analysis Summary \u2013 {dataset_name}",
                 fontweight="bold", fontsize=17,
                 fontproperties=fm.FontProperties(family="Times New Roman",
                                                   weight="bold", size=17))
    gs = fig.add_gridspec(3, 4, hspace=0.55, wspace=0.4)
    axes = [fig.add_subplot(gs[i // 4, i % 4]) for i in range(12)]
    panel = 0

    # ---- Panel 1: Dataset overview (head / shape as a text table) ----
    ax = axes[panel]; panel += 1
    ax.axis("off")
    overview_txt = (
        f"Shape: {df.shape[0]} rows x {df.shape[1]} cols\n"
        f"Numeric features: {len(numeric_cols)}\n"
        f"Categorical features: {len(categorical_cols)}\n"
        f"Missing cells: {int(df.isnull().sum().sum())}\n"
        f"Duplicate rows: {int(df.duplicated().sum())}"
    )
    ax.text(0.02, 0.9, overview_txt, va="top", ha="left",
            fontproperties=fm.FontProperties(family="Times New Roman", size=13),
            transform=ax.transAxes)
    _bold_axis_labels(ax, title="1. Dataset Overview")

    # ---- Panel 2: Statistical summary heat-table (mean/std/min/max) ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[["mean", "std", "min", "max"]]
        desc_norm = (desc - desc.min()) / (desc.max() - desc.min() + 1e-9)
        sns.heatmap(desc_norm.iloc[:8], annot=desc.iloc[:8].round(1), fmt="",
                    cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontsize": 8, "fontfamily": "Times New Roman"})
    _bold_axis_labels(ax, title="2. Statistical Summary")

    # ---- Panel 3: Missing value analysis ----
    ax = axes[panel]; panel += 1
    miss = df.isnull().mean().sort_values(ascending=False) * 100
    if miss.sum() == 0:
        ax.text(0.5, 0.5, "No Missing Values", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=14))
        ax.axis("off")
    else:
        miss[miss > 0].head(10).plot(kind="bar", ax=ax, color="#c0392b")
    _bold_axis_labels(ax, "Feature", "% Missing", "3. Missing Value Analysis")

    # ---- Panel 4: Class distribution / target distribution ----
    ax = axes[panel]; panel += 1
    if target_col is not None:
        if is_classification_target:
            df[target_col].value_counts().plot(kind="bar", ax=ax, color="#2980b9")
            _bold_axis_labels(ax, "Class", "Count", "4. Class Distribution")
        else:
            sns.histplot(df[target_col], kde=True, ax=ax, color="#2980b9")
            _bold_axis_labels(ax, target_col, "Frequency", "4. Target Distribution")
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "No target column supplied", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=12))
        _bold_axis_labels(ax, title="4. Target Distribution")

    # ---- Panel 5: Correlation matrix (heatmap) ----
    ax = axes[panel]; panel += 1
    corr_cols = numeric_cols[:10] if len(numeric_cols) > 10 else numeric_cols
    if len(corr_cols) >= 2:
        sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, ax=ax,
                    cbar=False, annot=len(corr_cols) <= 6, fmt=".2f",
                    annot_kws={"fontsize": 7})
    _bold_axis_labels(ax, title="5. Correlation Matrix")

    # ---- Panel 6: Feature distribution (histogram of 1st numeric feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.histplot(df[feat_a], kde=True, ax=ax, color="#27ae60")
    _bold_axis_labels(ax, feat_a if feat_a else "", "Frequency",
                       "6. Feature Distribution")

    # ---- Panel 7: Box plot (outlier detection) across numeric features ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        plot_cols = numeric_cols[:6]
        df_scaled = (df[plot_cols] - df[plot_cols].mean()) / (df[plot_cols].std() + 1e-9)
        sns.boxplot(data=df_scaled, ax=ax, color="#f39c12")
        ax.tick_params(axis="x", rotation=45)
    _bold_axis_labels(ax, "Feature", "Standardized Value", "7. Box Plot (Outliers)")

    # ---- Panel 8: Violin plot ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.violinplot(y=df[feat_a], ax=ax, color="#8e44ad")
    _bold_axis_labels(ax, "", feat_a if feat_a else "",
                       "8. Violin Plot")

    # ---- Panel 9: Scatter plot (feature 1 vs feature 2, hued by target) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None and feat_b is not None:
        hue = df[target_col] if (target_col and is_classification_target) else None
        sns.scatterplot(x=df[feat_a], y=df[feat_b],
                         hue=hue, ax=ax, palette="Set2", legend=False, s=18)
    _bold_axis_labels(ax, feat_a if feat_a else "", feat_b if feat_b else "",
                       "9. Scatter Plot")

    # ---- Panel 10: Q-Q plot (normality check on highest-variance feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        stats.probplot(df[feat_a].dropna(), dist="norm", plot=ax)
        ax.get_lines()[0].set_markerfacecolor("#2980b9")
        ax.get_lines()[0].set_markeredgecolor("#2980b9")
        ax.get_lines()[1].set_color("#c0392b")
    _bold_axis_labels(ax, "Theoretical Quantiles", "Sample Quantiles", "10. Q-Q Plot")

    # ---- Panel 11: KDE / density plot overlay of top numeric features ----
    ax = axes[panel]; panel += 1
    for c in kde_cols:
        sns.kdeplot(df[c], ax=ax, label=c, linewidth=1.5)
    if kde_cols:
        ax.legend(prop=fm.FontProperties(family="Times New Roman", size=9))
    _bold_axis_labels(ax, "Value", "Density", "11. KDE / Density Plot")

    # ---- Panel 12: Feature importance / variance plot ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        var = df[numeric_cols].var().sort_values(ascending=False).head(8)
        var.plot(kind="barh", ax=ax, color="#16a085")
        ax.invert_yaxis()
    _bold_axis_labels(ax, "Variance", "Feature", "12. Variance / Importance Plot")

    for ax in axes:
        _bold_axis_labels(ax)  # re-apply tick font in case a plotting call reset it

    saved = _save_eps(fig, save_path)
    if saved:
        print(f"[generate_eda_summary] Figure saved -> {saved} (600 DPI, EPS)")
    return fig


# --------------------------------------------------------------------------
# 3. GENERIC REGRESSION TRAIN/EVAL FUNCTION  (Section 4.2)
# --------------------------------------------------------------------------
def train_evaluate_regression(models: dict, X_train, X_test, y_train, y_test,
                               scale=False, verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of regression
    models on the same train/test split.

    Parameters
    ----------
    models : dict {name: sklearn-estimator}
    X_train, X_test, y_train, y_test : array-like
    scale : bool -> StandardScaler applied when True (fit on train only)
    verbose : bool -> print per-model metrics as they are computed

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by R2 desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = regression_performance_metrics(y_test, y_pred, model_name=name,
                                                   verbose=verbose, return_dict=True)
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("R2", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 4. GENERIC CLASSIFICATION TRAIN/EVAL FUNCTION  (Section 4.3)
# --------------------------------------------------------------------------
def train_evaluate_classification(models: dict, X_train, X_test, y_train, y_test,
                                   scale=False, average="weighted", verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of classification
    models on the same train/test split.

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by Accuracy desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = None
        if hasattr(model, "predict_proba"):
            try:
                y_proba = model.predict_proba(X_test)
            except Exception:
                y_proba = None
        metrics = classification_performance_metrics(
            y_test, y_pred, y_proba=y_proba, model_name=name,
            average=average, verbose=verbose, return_dict=True, plot=False
        )
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("Accuracy", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 5. GENERIC REGRESSION METRICS FUNCTION  (Section 4.4)
# --------------------------------------------------------------------------
def regression_performance_metrics(y_true, y_pred, model_name="Model",
                                    verbose=True, return_dict=False):
    """
    Computes and displays ALL standard regression performance metrics:
    MAE, MSE, RMSE, R2, Adjusted R2 (n only), MAPE.
    """
    from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                                  r2_score, mean_absolute_percentage_error)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    if verbose:
        print(f"--- Regression Metrics: {model_name} ---")
        print(f"  MAE  : {mae:.4f}")
        print(f"  MSE  : {mse:.4f}")
        print(f"  RMSE : {rmse:.4f}")
        print(f"  R2   : {r2:.4f}")
        print(f"  MAPE : {mape:.2f}%\n")

    result = {"Model": model_name, "MAE": mae, "MSE": mse,
              "RMSE": rmse, "R2": r2, "MAPE(%)": mape}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


# --------------------------------------------------------------------------
# 6. GENERIC CLASSIFICATION METRICS FUNCTION  (Section 4.5)
# --------------------------------------------------------------------------
def classification_performance_metrics(y_true, y_pred, y_proba=None,
                                        model_name="Model", average="weighted",
                                        verbose=True, return_dict=False,
                                        plot=True, save_path=None):
    """
    Computes and displays ALL standard classification performance metrics:
    Accuracy, Precision, Recall, F1-score, ROC-AUC (binary/multiclass ovr),
    and (optionally) plots the confusion matrix using the mandatory lab
    formatting (Times New Roman, bold 15pt axis labels).
    """
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, roc_auc_score, confusion_matrix)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)

    roc_auc = np.nan
    if y_proba is not None:
        try:
            n_classes = y_proba.shape[1]
            if n_classes == 2:
                roc_auc = roc_auc_score(y_true, y_proba[:, 1])
            else:
                roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr",
                                         average=average)
        except Exception:
            roc_auc = np.nan

    if verbose:
        print(f"--- Classification Metrics: {model_name} ---")
        print(f"  Accuracy  : {acc:.4f}")
        print(f"  Precision : {prec:.4f}")
        print(f"  Recall    : {rec:.4f}")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  ROC-AUC   : {roc_auc:.4f}" if not np.isnan(roc_auc) else "  ROC-AUC   : N/A")
        print()

    if plot:
        set_plot_style()
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontfamily": "Times New Roman", "fontsize": 13})
        _bold_axis_labels(ax, "Predicted Label", "True Label",
                           f"Confusion Matrix \u2013 {model_name}")
        _save_eps(fig, save_path)

    result = {"Model": model_name, "Accuracy": acc, "Precision": prec,
              "Recall": rec, "F1-score": f1, "ROC-AUC": roc_auc}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


In [1]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                      RandomizedSearchCV, KFold, cross_val_score,
                                      cross_validate)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


warnings.filterwarnings("ignore")
RANDOM_STATE = 42

FIG_DIR = "figures"
RES_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

set_plot_style()


## 1. LOAD DATASET

In [2]:
DATA_PATH = "loan_data.csv"  # <-- upload this file to the notebook's working directory
df_raw = pd.read_csv(DATA_PATH)
print("Raw dataset shape:", df_raw.shape)
print(df_raw.isnull().sum())

df = df_raw.drop(columns=["Loan_ID"]).copy()

# Drop rows where the TARGET itself is missing -- cannot train/evaluate on
# an unknown target value.
n_before = len(df)
df = df.dropna(subset=["LoanAmount"]).reset_index(drop=True)
print(f"\nDropped {n_before - len(df)} rows with missing target (LoanAmount).")
print("Working dataset shape:", df.shape)

Raw dataset shape: (614, 13)
Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

Dropped 22 rows with missing target (LoanAmount).
Working dataset shape: (592, 12)


## 2. HANDLE MISSING VALUES (features only, target already clean)

In [3]:
categorical_cols = ["Gender", "Married", "Dependents", "Education",
                     "Self_Employed", "Property_Area", "Loan_Status"]
numeric_cols = ["ApplicantIncome", "CoapplicantIncome", "Loan_Amount_Term",
                 "Credit_History"]

for c in categorical_cols:
    df[c] = df[c].fillna(df[c].mode()[0])
for c in numeric_cols:
    df[c] = df[c].fillna(df[c].median())

print("\nMissing values after imputation:", int(df.isnull().sum().sum()))


Missing values after imputation: 0


## 3. ENCODE CATEGORICAL VARIABLES

In [4]:
df_encoded = df.copy()
# Dependents has a "3+" category -> map to numeric ordinal 0/1/2/3
df_encoded["Dependents"] = df_encoded["Dependents"].replace("3+", "3").astype(int)
df_encoded = pd.get_dummies(
    df_encoded,
    columns=["Gender", "Married", "Education", "Self_Employed",
             "Property_Area", "Loan_Status"],
    drop_first=True
)
# Coerce any bool dummy columns to int for downstream numeric operations
bool_cols = df_encoded.select_dtypes(include="bool").columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print("\nEncoded dataset shape:", df_encoded.shape)
print("Encoded columns:", df_encoded.columns.tolist())


Encoded dataset shape: (592, 13)
Encoded columns: ['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Gender_Male', 'Married_Yes', 'Education_Not Graduate', 'Self_Employed_Yes', 'Property_Area_Semiurban', 'Property_Area_Urban', 'Loan_Status_Y']


## 4. EDA (reusable function from Experiment 1) -- run on the pre-encoding,

In [5]:
#    imputed dataframe so categorical distributions remain human-readable.
# --------------------------------------------------------------------------
generate_eda_summary(
    df, target_col="LoanAmount", dataset_name="Loan Amount Prediction",
    save_path=f"{FIG_DIR}/eda_loan.eps"
)
plt.close("all")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[generate_eda_summary] Figure saved -> figures/eda_loan.eps (600 DPI, EPS)


## 5. ADDITIONAL REQUIRED VISUALIZATIONS

In [6]:
# 5a. Target variable distribution
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(df["LoanAmount"], kde=True, ax=ax, color="#2980b9")
_bold_axis_labels(ax, "Loan Amount (thousands)", "Frequency", "Target Variable Distribution")
_save_eps(fig, f"{FIG_DIR}/target_distribution.eps")
plt.close(fig)

# 5b. Feature vs target scatter plots (top 2 numeric features)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(x=df["ApplicantIncome"], y=df["LoanAmount"], ax=axes[0],
                 color="#27ae60", s=20)
_bold_axis_labels(axes[0], "Applicant Income", "Loan Amount",
                   "Applicant Income vs Loan Amount")
sns.scatterplot(x=df["CoapplicantIncome"], y=df["LoanAmount"], ax=axes[1],
                 color="#8e44ad", s=20)
_bold_axis_labels(axes[1], "Coapplicant Income", "Loan Amount",
                   "Coapplicant Income vs Loan Amount")
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/feature_vs_target_scatter.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


## 6. TRAIN / TEST SPLIT + FEATURE SCALING

In [7]:
X = df_encoded.drop(columns=["LoanAmount"]).values
y = df_encoded["LoanAmount"].values
feature_names = df_encoded.drop(columns=["LoanAmount"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 7. BASELINE LINEAR REGRESSION (via reusable train_evaluate_regression)

In [8]:
baseline_models = {"Linear Regression": LinearRegression()}
baseline_results_df, baseline_fitted = train_evaluate_regression(
    baseline_models, X_train_scaled, X_test_scaled, y_train, y_test, scale=False
)
print("\n=== Baseline Linear Regression ===")
print(baseline_results_df)

--- Regression Metrics: Linear Regression ---
  MAE  : 36.8850
  MSE  : 3335.4303
  RMSE : 57.7532
  R2   : 0.0952
  MAPE : 35.58%


=== Baseline Linear Regression ===
                         MAE          MSE       RMSE        R2    MAPE(%)
Model                                                                    
Linear Regression  36.884969  3335.430337  57.753185  0.095202  35.580091


## 8. HYPERPARAMETER TUNING: RIDGE, LASSO, ELASTIC NET

In [9]:
#    (5-fold CV, both GridSearchCV and RandomizedSearchCV)
# --------------------------------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search_spaces = {
    "Ridge Regression": (Ridge(random_state=RANDOM_STATE),
                          {"alpha": [0.01, 0.1, 1, 10, 100]}),
    "Lasso Regression": (Lasso(random_state=RANDOM_STATE, max_iter=10000),
                          {"alpha": [0.001, 0.01, 0.1, 1, 10]}),
    "Elastic Net Regression": (ElasticNet(random_state=RANDOM_STATE, max_iter=10000),
                                {"alpha": [0.01, 0.1, 1, 10],
                                 "l1_ratio": [0.2, 0.5, 0.8]}),
}

tuning_summary_rows = []
best_models = {"Linear Regression": baseline_fitted["Linear Regression"]}

for name, (estimator, grid) in search_spaces.items():
    # GridSearchCV
    t0 = time.perf_counter()
    gs = GridSearchCV(estimator, grid, cv=kf, scoring="r2", n_jobs=-1)
    gs.fit(X_train_scaled, y_train)
    grid_time = time.perf_counter() - t0

    # RandomizedSearchCV
    n_combinations = np.prod([len(v) for v in grid.values()])
    n_iter = min(10, int(n_combinations))
    t0 = time.perf_counter()
    rs = RandomizedSearchCV(estimator, grid, cv=kf, scoring="r2",
                             n_iter=n_iter, random_state=RANDOM_STATE, n_jobs=-1)
    rs.fit(X_train_scaled, y_train)
    random_time = time.perf_counter() - t0

    # Use GridSearchCV's result as the "official" tuned model (exhaustive,
    # deterministic); RandomizedSearchCV result reported for comparison only.
    tuning_summary_rows.append({
        "Model": name,
        "Search Method": "GridSearchCV",
        "Best Parameters": str(gs.best_params_),
        "Best CV R2": gs.best_score_,
        "Execution Time (s)": grid_time,
    })
    tuning_summary_rows.append({
        "Model": name,
        "Search Method": "RandomizedSearchCV",
        "Best Parameters": str(rs.best_params_),
        "Best CV R2": rs.best_score_,
        "Execution Time (s)": random_time,
    })
    best_models[name] = gs.best_estimator_

tuning_summary_df = pd.DataFrame(tuning_summary_rows).set_index(["Model", "Search Method"])
tuning_summary_df.to_csv(f"{RES_DIR}/hyperparameter_tuning_summary.csv")
print("\n=== Hyperparameter Tuning Summary ===")
print(tuning_summary_df)


=== Hyperparameter Tuning Summary ===
                                                         Best Parameters  \
Model                  Search Method                                       
Ridge Regression       GridSearchCV                       {'alpha': 100}   
                       RandomizedSearchCV                 {'alpha': 100}   
Lasso Regression       GridSearchCV                        {'alpha': 10}   
                       RandomizedSearchCV                  {'alpha': 10}   
Elastic Net Regression GridSearchCV        {'alpha': 1, 'l1_ratio': 0.8}   
                       RandomizedSearchCV  {'l1_ratio': 0.8, 'alpha': 1}   

                                           Best CV R2  Execution Time (s)  
Model                  Search Method                                       
Ridge Regression       GridSearchCV          0.371431            0.029575  
                       RandomizedSearchCV    0.371431            0.028415  
Lasso Regression       GridSearchCV          0.3

## 9. 5-FOLD CROSS-VALIDATION PERFORMANCE (all 4 models, on training data)

In [10]:
scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2",
}

cv_rows = []
for name, model in best_models.items():
    cv_res = cross_validate(model, X_train_scaled, y_train, cv=kf, scoring=scoring)
    mae = -cv_res["test_MAE"].mean()
    mse = -cv_res["test_MSE"].mean()
    rmse = np.sqrt(mse)
    r2 = cv_res["test_R2"].mean()
    cv_rows.append({"Model": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2})

cv_performance_df = pd.DataFrame(cv_rows).set_index("Model")
cv_performance_df.to_csv(f"{RES_DIR}/cross_validation_performance.csv")
print("\n=== 5-Fold Cross-Validation Performance ===")
print(cv_performance_df)


=== 5-Fold Cross-Validation Performance ===
                              MAE          MSE       RMSE        R2
Model                                                              
Linear Regression       43.105913  5508.550230  74.219608  0.310890
Ridge Regression        44.084260  4935.283002  70.251569  0.371431
Lasso Regression        46.230622  5281.512734  72.674017  0.331080
Elastic Net Regression  43.881942  4944.289781  70.315644  0.370701


## 10. TEST SET PERFORMANCE (all 4 models, reusable metrics function)

In [11]:
test_rows = []
test_predictions = {}
train_times = {}
for name, model in best_models.items():
    t0 = time.perf_counter()
    model.fit(X_train_scaled, y_train)
    train_t = time.perf_counter() - t0
    y_pred = model.predict(X_test_scaled)
    test_predictions[name] = y_pred
    train_times[name] = train_t
    metrics = regression_performance_metrics(y_test, y_pred, model_name=name,
                                               verbose=True, return_dict=True)
    metrics["Training Time (s)"] = train_t
    test_rows.append(metrics)

test_performance_df = pd.DataFrame(test_rows).set_index("Model")
test_performance_df.to_csv(f"{RES_DIR}/test_set_performance.csv")
print("\n=== Test Set Performance ===")
print(test_performance_df)

best_model_name = test_performance_df["R2"].idxmax()
print(f"\nBest model on test R2: {best_model_name}")

--- Regression Metrics: Linear Regression ---
  MAE  : 36.8850
  MSE  : 3335.4303
  RMSE : 57.7532
  R2   : 0.0952
  MAPE : 35.58%

--- Regression Metrics: Ridge Regression ---
  MAE  : 37.5460
  MSE  : 3159.5508
  RMSE : 56.2099
  R2   : 0.1429
  MAPE : 37.10%

--- Regression Metrics: Lasso Regression ---
  MAE  : 37.7959
  MSE  : 3176.8000
  RMSE : 56.3631
  R2   : 0.1382
  MAPE : 39.05%

--- Regression Metrics: Elastic Net Regression ---
  MAE  : 37.4300
  MSE  : 3151.2142
  RMSE : 56.1357
  R2   : 0.1452
  MAPE : 37.16%


=== Test Set Performance ===
                              MAE          MSE       RMSE        R2  \
Model                                                                 
Linear Regression       36.884969  3335.430337  57.753185  0.095202   
Ridge Regression        37.546000  3159.550821  56.209882  0.142912   
Lasso Regression        37.795921  3176.799969  56.363108  0.138233   
Elastic Net Regression  37.430050  3151.214157  56.135676  0.145174   

            

## 11. VISUALIZATIONS: Predicted vs Actual, Residuals, Coefficients

In [12]:
# 11a. Predicted vs Actual (2x2 grid, all four models)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for ax, (name, y_pred) in zip(axes, test_predictions.items()):
    ax.scatter(y_test, y_pred, alpha=0.5, s=20, color="#2980b9")
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, linestyle="--", color="#c0392b", linewidth=1.5)
    _bold_axis_labels(ax, "Actual Loan Amount", "Predicted Loan Amount", name)
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/predicted_vs_actual.eps")
plt.close(fig)

# 11b. Residual plots (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for ax, (name, y_pred) in zip(axes, test_predictions.items()):
    residuals = y_test - y_pred
    ax.scatter(y_pred, residuals, alpha=0.5, s=20, color="#e67e22")
    ax.axhline(0, linestyle="--", color="#c0392b", linewidth=1.5)
    _bold_axis_labels(ax, "Predicted Loan Amount", "Residual", name)
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/residual_plots.eps")
plt.close(fig)

# 11c. Training error vs validation error (learning-curve-style, via CV folds)
train_val_rows = []
for name, model in best_models.items():
    cv_res = cross_validate(model, X_train_scaled, y_train, cv=kf,
                             scoring="neg_mean_squared_error",
                             return_train_score=True)
    train_rmse = np.sqrt(-cv_res["train_score"]).mean()
    val_rmse = np.sqrt(-cv_res["test_score"]).mean()
    train_val_rows.append({"Model": name, "Train RMSE": train_rmse, "Validation RMSE": val_rmse})
train_val_df = pd.DataFrame(train_val_rows).set_index("Model")
train_val_df.to_csv(f"{RES_DIR}/train_vs_validation_error.csv")
print("\n=== Training vs Validation Error ===")
print(train_val_df)

fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(len(train_val_df))
width = 0.35
ax.bar(x_pos - width/2, train_val_df["Train RMSE"], width, label="Training RMSE", color="#2980b9")
ax.bar(x_pos + width/2, train_val_df["Validation RMSE"], width, label="Validation RMSE", color="#c0392b")
ax.set_xticks(x_pos)
ax.set_xticklabels(train_val_df.index, rotation=20, ha="right")
ax.legend()
_bold_axis_labels(ax, "Model", "RMSE", "Training Error vs Validation Error")
_save_eps(fig, f"{FIG_DIR}/train_vs_validation_error.eps")
plt.close(fig)

# 11d. Coefficient comparison bar plot
coef_data = {}
for name, model in best_models.items():
    coef_data[name] = model.coef_
coef_df = pd.DataFrame(coef_data, index=feature_names)
coef_df.to_csv(f"{RES_DIR}/coefficient_comparison.csv")
print("\n=== Coefficient Comparison (first 5 features) ===")
print(coef_df.head())

top_features = coef_df.abs().sum(axis=1).sort_values(ascending=False).head(8).index
fig, ax = plt.subplots(figsize=(10, 6))
coef_df.loc[top_features].plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.legend(prop=fm.FontProperties(family="Times New Roman", size=11))
_bold_axis_labels(ax, "Coefficient Value", "Feature", "Coefficient Comparison (Top 8 Features)")
_save_eps(fig, f"{FIG_DIR}/coefficient_comparison.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== Training vs Validation Error ===
                        Train RMSE  Validation RMSE
Model                                              
Linear Regression        66.090963        73.522699
Ridge Regression         67.596436        69.944944
Lasso Regression         69.990230        72.300631
Elastic Net Regression   67.336192        70.021945

=== Coefficient Comparison (first 5 features) ===
                   Linear Regression  Ridge Regression  Lasso Regression  \
Dependents                  5.044221          5.766139          0.000000   
ApplicantIncome            53.951546         43.826365         44.979158   
CoapplicantIncome          21.968223         16.999975         11.788210   
Loan_Amount_Term            7.631447          5.589587          0.000000   
Credit_History              3.316734          2.177475          0.000000   

                   Elastic Net Regression  
Dependents                       5.306926  
ApplicantIncome                 43.720817  
Coapplican

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


## 12. REGULARIZATION PATH / OVERFITTING-UNDERFITTING ANALYSIS

In [13]:
# Ridge: effect of alpha on train/val RMSE
ridge_alphas = [0.01, 0.1, 1, 10, 100]
ridge_path_rows = []
for a in ridge_alphas:
    model = Ridge(alpha=a, random_state=RANDOM_STATE)
    cv_res = cross_validate(model, X_train_scaled, y_train, cv=kf,
                             scoring="neg_mean_squared_error", return_train_score=True)
    ridge_path_rows.append({
        "alpha": a,
        "Train RMSE": np.sqrt(-cv_res["train_score"]).mean(),
        "Validation RMSE": np.sqrt(-cv_res["test_score"]).mean(),
    })
ridge_path_df = pd.DataFrame(ridge_path_rows).set_index("alpha")
ridge_path_df.to_csv(f"{RES_DIR}/ridge_alpha_path.csv")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(ridge_path_df.index, ridge_path_df["Train RMSE"], marker="o",
        label="Training RMSE", color="#2980b9")
ax.plot(ridge_path_df.index, ridge_path_df["Validation RMSE"], marker="s",
        label="Validation RMSE", color="#c0392b")
ax.set_xscale("log")
ax.legend()
_bold_axis_labels(ax, "Alpha (log scale)", "RMSE", "Ridge: Effect of Regularization Strength")
_save_eps(fig, f"{FIG_DIR}/ridge_alpha_path.eps")
plt.close(fig)

# Lasso: number of non-zero coefficients vs alpha (feature sparsity)
lasso_alphas = [0.001, 0.01, 0.1, 1, 10]
lasso_sparsity_rows = []
for a in lasso_alphas:
    model = Lasso(alpha=a, random_state=RANDOM_STATE, max_iter=10000)
    model.fit(X_train_scaled, y_train)
    n_nonzero = int(np.sum(model.coef_ != 0))
    lasso_sparsity_rows.append({"alpha": a, "Non-zero Coefficients": n_nonzero})
lasso_sparsity_df = pd.DataFrame(lasso_sparsity_rows).set_index("alpha")
lasso_sparsity_df.to_csv(f"{RES_DIR}/lasso_sparsity.csv")
print("\n=== Lasso Feature Sparsity vs Alpha ===")
print(lasso_sparsity_df)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(lasso_sparsity_df.index, lasso_sparsity_df["Non-zero Coefficients"],
        marker="o", color="#8e44ad")
ax.set_xscale("log")
_bold_axis_labels(ax, "Alpha (log scale)", "Non-zero Coefficients",
                   "Lasso: Feature Sparsity vs Regularization Strength")
_save_eps(fig, f"{FIG_DIR}/lasso_sparsity.eps")
plt.close(fig)

print("\nAll figures saved under:", os.path.abspath(FIG_DIR))
print("All result tables saved under:", os.path.abspath(RES_DIR))
print("\nDone.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== Lasso Feature Sparsity vs Alpha ===
        Non-zero Coefficients
alpha                        
0.001                      12
0.010                      12
0.100                      12
1.000                      10
10.000                      3



All figures saved under: /home/claude/notebooks/figures
All result tables saved under: /home/claude/notebooks/results

Done.
